# Task 3a: YCSEP preparation and Parakeet fine-tuning

Status: metadata preparation and a two-step GPU execution test have passed. The full experiment and its interpretation remain pending. This notebook can run preparation and training, or inspect an existing recorded run without retraining.

Hypothesis: reliable, diverse Singapore conversational speech can improve domain adaptation per GPU-hour. Improvement is a hypothesis, not an assumed result.

In [ ]:
import json, os, sys
from pathlib import Path
root = Path.cwd().resolve()
if root.name == 'asr-train':
    root = root.parent
sys.path.insert(0, str(root / 'asr-train'))
from prepare_metadata import prepare
source = Path(os.environ.get('YCSEP_CSV', str(root / 'test_docs/test/runtime/local-prep-0316/data/YCSEP_static.csv')))
output = Path(os.environ.get('YCSEP_AUDIT', str(root / 'test_docs/test/runtime/metadata-audit')))
print('Source:', source)
print('Output:', output)

## Freeze the split before curation

All TDK source-video IDs and audio URLs are held out. Non-TDK videos receive a deterministic hash-based train/validation split, seed 2026, expected validation fraction 10%. The realized fraction need not equal 10%. Video grouping reduces adjacent-clip leakage but does not prove speaker independence. Near-duplicate audio requires a later acoustic check.

The CPU candidate policy retains original transcripts and Singlish. Invalid timestamps, empty lexical text, invalid URLs, duplicate URLs and durations outside 0.3-20 seconds are excluded from the pilot candidate pool. Sparse/dense text, fillers and repeated characters remain flags. Validation receives only objective validity checks; later learned quality thresholds must not redefine it to improve reported performance.

In [ ]:
assert source.exists(), 'Set YCSEP_CSV to the supplied YCSEP_static.csv'
audit = prepare(source, output)
audit

## Inspect duration, text and overlap

Segment-hours may count overlapping intervals more than once. The sweep-line overlap report separately measures union time and time covered by at least two annotation intervals. Annotation overlap is not proof of simultaneous speech. TDK metadata is reported for coverage, not used to tune quality rules.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(list(audit['channel_segment_hours']), list(audit['channel_segment_hours'].values()))
axes[0].set_xlabel('Annotated segment-hours (not unique audio hours)')
axes[1].bar(list(audit['duration_buckets']), list(audit['duration_buckets'].values()))
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Segments')
fig.tight_layout()
fig.savefig(output / 'metadata_overview.png', dpi=160)
plt.show()

## Audio preparation and feature extraction

Next: sample a diverse ~10-hour pilot from training candidates, freeze a non-TDK validation sample, download on CPU, validate nonempty finite waveforms and actual duration, and convert to mono 16 kHz WAV. Candidate JSONL files are not yet NeMo training manifests. Audio-ready manifests require audio_filepath, measured duration and original text, plus source-video provenance.

Retain Parakeet's pretrained SentencePiece tokenizer and acoustic preprocessor. Log the loaded tokenizer vocabulary and preprocessor configuration rather than guessing feature parameters. Inspect Singlish tokenization without retraining the tokenizer. Use VAD only on suspicious sampled cases initially; defer MOS models unless they change a demonstrated data decision.

## Planned PyTorch/NeMo experiment

E0: base model WER/CER on frozen non-TDK validation. E1: full fine-tuning on a ~10-hour pilot, initially AdamW learning rate 1e-5, weight decay 0.001, batch size 8, gradient accumulation 4, BF16 on supported GPU, gradient clipping 1.0. These are starting values, not optimized settings. Profile memory and steps/second before setting the final duration budget. Reuse model-compatible SpecAugment; do not add unverified augmentation defaults.

Validate every 200 microbatches (50 optimizer updates at accumulation 4), with at most 1,200 updates and a 120-minute training budget. Select the lowest non-TDK validation WER. Save best weights, not optimizer restart state, to bound artifact size. Record loss, WER, actual data hours, optimizer, seed, package versions and elapsed time. No parameter search or larger-data expansion is planned for this deadline; no TDK-based model selection.

Final artifact: parakeet-tdt-0.6b-v3-ycsep.nemo. Training implementation, executed loss/WER plots and evidence-based interpretation remain to be completed.

## Interpretation to complete after execution

Compare E0 and E1 on the same validation rows. Decreasing training loss alone does not demonstrate generalization. Inspect worsening validation WER, plateau, corrupted batches and subgroup regressions before scaling. Report failed audio coverage separately. Never replace a missing experiment with a fabricated curve.

References: [Parakeet model card](https://huggingface.co/nvidia/parakeet-tdt-0.6b-v3), [JiWER](https://jitsi.github.io/jiwer/).

## Prepare the frozen audio manifests
The original tokenizer (8,192 tokens) and pretrained acoustic preprocessor are retained. PCM16 mono 16 kHz conversion is performed on CPU. Equal annotated hours per non-TDK channel and round-robin source videos bound the pilot while retaining diversity. Duration checks do not establish transcript correctness. The local execution test exposed a TDT CUDA dependency problem; the training Docker environment pins Numba and NVIDIA numba-cuda and has passed actual optimization.

In [ ]:
from argparse import Namespace
from select_pilot import main as select_pilot
from prepare_audio import prepare as prepare_audio
selection = Path(os.environ.get('YCSEP_SELECTION', str(root / 'test_docs/test/runtime/pilot-selection')))
audio = Path(os.environ.get('YCSEP_AUDIO', str(root / 'test_docs/test/runtime/pilot-audio')))
if not (selection / 'selection.json').exists():
    select_pilot(Namespace(audit=str(output), output=str(selection), train_hours=10, validation_hours=2))
for split in ['validation', 'train']:
    if not (audio / f'{split}.jsonl').exists():
        prepare_audio(selection / f'{split}-selected.jsonl', audio, split, workers=16)
    report = json.loads((audio / f'{split}-result.json').read_text())
    assert not report['failures'], f'Unresolved audio failures in {split}'
    print({k:v for k,v in report.items() if k != 'failures'})

In [ ]:
from train_parakeet import train
experiment = Path(os.environ.get('YCSEP_EXPERIMENT', str(root / 'test_docs/test/runtime/training-final')))
result_path = experiment / 'training-result.json'
if result_path.exists():
    print('Inspecting a recorded run; no new training is being performed.')
    result = json.loads(result_path.read_text())
else:
    result = train(audio / 'train.jsonl', audio / 'validation.jsonl', experiment)
assert (experiment / 'parakeet-tdt-0.6b-v3-ycsep.nemo').exists()
result

In [ ]:
import pandas as pd
metrics_path = sorted((experiment / 'training').glob('version_*/metrics.csv'))[-1]
metrics = pd.read_csv(metrics_path)
loss_column = next(c for c in ['train_loss', 'train_loss_step', 'loss'] if c in metrics.columns)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
loss = metrics.dropna(subset=[loss_column])
validation = metrics.dropna(subset=['val_wer'])
axes[0].plot(loss['step'], loss[loss_column], linewidth=1)
axes[0].set(xlabel='Optimizer step', ylabel='Training loss')
axes[1].plot(validation['step'], validation['val_wer'], marker='o')
axes[1].set(xlabel='Optimizer step', ylabel='Non-TDK validation WER')
fig.tight_layout()
fig.savefig(experiment / 'training-curves.png', dpi=160)
plt.show()
baseline = result['baseline_validation'][0]['val_wer']
print(f"Base native validation WER: {baseline:.4f}; selected FT: {result['best_validation_wer']:.4f}")
print(f"Training time: {result['elapsed_training_seconds']/60:.1f} minutes; updates: {result['optimizer_steps']}")

## Interpretation boundary
The printed comparison uses NeMo's native validation WER consistently for E0 and E1. Task 3b uses a documented Unicode-normalized WER/CER comparison for both models on TDK. These differently normalized metrics should not be compared directly. Full-run curve interpretation will be added after execution; the two-step smoke test only proves executable training and export, not improved recognition.

[NeMo TDT CUDA compatibility issue](https://github.com/NVIDIA-NeMo/Speech/issues/14941).